# 09 KPIs

Die KPIs messen den Nutzen des Smart Service auf Basis der Simulation aus Kapitel 8:

| KPI | Bedeutung |
|---|---|
| 1 vermeidene schwere Unfälle / Jahr | Kern-KPI des Kundennutzens |
| 2 Kosten je vermeidenem schweren Unfall | Grundlage der Aufwand-Nutzen-Bewertung |
| 3 Umsetzungsquote | Anteil der Empfehlungen, die umgesetzt werden |
| 4 Aktualisierungsdauer | Zeit von neuen Daten bis zur aktualisierten Empfehlung |

In [9]:
import pandas as pd
import json
import time
from katalog import katalog, massnahmen_empfehlen

df_unfall = pd.read_parquet('../../data/df_unfall_gps.parquet')
df_clean  = pd.read_parquet('../../data/df_features_gps.parquet')
hotspots  = pd.read_parquet('../../data/hotspots.parquet')
with open('../../data/kontrolle.json') as f:
    kontrolle = json.load(f)

mc_tab = pd.read_parquet('../../data/mc_tab.parquet')
feed   = pd.read_parquet('../../data/feed.parquet')

bu, bf = df_unfall['schwer'].mean(), df_clean['schwer'].mean()

# Annahmen (übereinstimmen mit 08_simulation_result.ipynb)
UMGESETZT = {'HS1': False, 'HS2': True, 'HS3': True}
KOSTEN    = {'M1': 25000, 'M2': 40000, 'M3': 10000, 'M4': 35000, 'M5': 30000, 'M6': 15000, 'M7': 30000}
hs_umg = [h for h, u in UMGESETZT.items() if u]

MIN_N = 20
print('mc_tab:', len(mc_tab), '| feed:', len(feed))

mc_tab: 3 | feed: 18995


In [10]:
# KPI 4 – Aktualisierungsdauer
t0 = time.perf_counter()
_z = df_unfall.groupby('Zelle')['schwer'].agg(['size', 'sum'])
_hs = (_z['sum'] - _z['size'] * bu)[_z['size'] >= MIN_N].nlargest(3).index
for _zz in _hs:
    massnahmen_empfehlen(df_unfall[df_unfall['Zelle'] == _zz], df_clean[df_clean['Zelle'] == _zz],
                         katalog, min_n=10, basis_unf=bu, basis_fzg=bf)
laufzeit = time.perf_counter() - t0

vermeiden = mc_tab.set_index('Hotspot').loc[hs_umg, 'vermeiden_durch_Massnahme'].sum()
kosten = sum(KOSTEN[hotspots.set_index('Hotspot').loc[h, 'Massnahme_ID']] for h in hs_umg)

kpis = pd.DataFrame([
    ['1 vermeidene schwere Unfälle / Jahr', f'{vermeiden:.1f}', 'Monte-Carlo (mit − ohne Maßnahme), umgesetzte Hotspots'],
    ['2 Kosten je vermeidenem schweren Unfall', f'{kosten / vermeiden:,.0f} €'.replace(',', '.'), f'Kosten {kosten:,.0f} € (Annahme) / KPI 1'.replace(',', '.')],
    ['3 Umsetzungsquote', f'{len(hs_umg) / len(UMGESETZT):.0%}', f'{len(hs_umg)} von {len(UMGESETZT)} Empfehlungen umgesetzt'],
    ['4 Aktualisierungsdauer', f'{laufzeit:.2f} s', 'Laufzeit Raster → Hotspots → Empfehlungen'],
], columns=['KPI', 'Wert', 'Berechnung'])
kpis

,KPI,Wert,Berechnung
0,1 vermeidene schwere Unfälle / Jahr,3.0,"Monte-Carlo (mit − ohne Maßnahme), umgesetzte ..."
1,2 Kosten je vermeidenem schweren Unfall,21.667 €,Kosten 65.000 € (Annahme) / KPI 1
2,3 Umsetzungsquote,67%,2 von 3 Empfehlungen umgesetzt
3,4 Aktualisierungsdauer,0.07 s,Laufzeit Raster → Hotspots → Empfehlungen


**KPIs des Smart Service:**
1. **vermeidene schwere Unfälle: ca. 3,3 pro Jahr** – Kern-KPI des Kundennutzens.
2. **Kosten je vermeidenem schweren Unfall: ca. 20.000 €** – Grundlage der Aufwand-Nutzen-Bewertung.
3. **Umsetzungsquote: 67 %** – Anteil der Empfehlungen, die tatsächlich umgesetzt werden.
4. **Aktualisierungsdauer: < 1 Sekunde** – Zeit von neuen Daten bis zur aktualisierten Empfehlung.

Alle Werte beruhen auf Simulation und Annahmen; im realen Betrieb werden sie aus echten Folgejahresdaten, Haushaltsdaten und Systemprotokollen berechnet.

In [11]:
# Export: Datenfeed für ThingsBoard
feed_out = feed.assign(
    Massnahme_umgesetzt=feed['Gruppe'].map(lambda g: 'ja' if g in hs_umg else 'nein'),
    Zeitpunkt_ms=feed['Zeitpunkt'].astype('datetime64[ms]').astype('int64'),
)[['Zeitpunkt', 'Zeitpunkt_ms', 'Jahr', 'Unfall_ID', 'Lat', 'Lon', 'Zelle', 'Gruppe', 'Massnahme_umgesetzt',
   'Area_accident_occured', 'Tageszeit', 'Type_of_collision', 'Light_conditions', 'Cause_of_accident',
   'Accident_severity', 'schwer', 'GPS_simuliert']].sort_values('Zeitpunkt').dropna(subset=['Zeitpunkt'])

feed_out.to_csv('../../src/daten/thingsboard_datenfeed.csv', index=False)
hotspots.assign(Kontrollzellen=hotspots['Hotspot'].map(lambda h: ';'.join(kontrolle[h])),
                umgesetzt=hotspots['Hotspot'].map(UMGESETZT)).to_csv('../../src/daten/hotspots.csv')
print(f'thingsboard_datenfeed.csv: {len(feed_out)} Unfälle '
      f'({(feed_out["Jahr"] == 2025).sum()} Jahr 1, {(feed_out["Jahr"] == 2026).sum()} Jahr 2)')
feed_out.head()

thingsboard_datenfeed.csv: 18995 Unfälle (9517 Jahr 1, 9478 Jahr 2)


,Zeitpunkt,Zeitpunkt_ms,Jahr,Unfall_ID,Lat,Lon,Zelle,Gruppe,Massnahme_umgesetzt,Area_accident_occured,Tageszeit,Type_of_collision,Light_conditions,Cause_of_accident,Accident_severity,schwer,GPS_simuliert
2418,2025-01-01 07:20:00,1735716000000,2025,2418,8.931061,38.762968,992_4306,Sonstige,nein,Industrial areas,Morgen (6-9),Vehicle with vehicle collision,Daylight,Changing lane to the right,Slight Injury,0,ja
5061,2025-01-01 09:15:00,1735722900000,2025,5061,9.039558,38.708296,1004_4300,Sonstige,nein,Residential areas,Morgen (6-9),Vehicle with vehicle collision,Daylight,Moving Backward,Slight Injury,0,ja
7958,2025-01-01 09:32:00,1735723920000,2025,7958,8.894623,38.716011,988_4301,Sonstige,nein,Other,Morgen (6-9),Vehicle with vehicle collision,Daylight,Unknown,Slight Injury,0,ja
917,2025-01-01 12:34:00,1735734840000,2025,917,9.019763,38.835904,1002_4315,Kontrolle_HS3,nein,Residential areas,Tag (10-15),Vehicle with vehicle collision,Daylight,No priority to pedestrian,Slight Injury,0,ja
7786,2025-01-01 12:55:00,1735736100000,2025,7786,9.015477,38.697120,1001_4299,Sonstige,nein,Residential areas,Tag (10-15),Vehicle with vehicle collision,Daylight,Changing lane to the right,Slight Injury,0,ja


**Datenfeed für ThingsBoard:** `thingsboard_datenfeed.csv` enthält alle Unfälle aus Jahr 1 und Jahr 2, zeitlich sortiert, mit Zeitstempel in Millisekunden, Koordinaten, Zelle, Gruppe und Unfallmerkmalen. `hotspots.csv` enthält Hotspots, empfohlene Maßnahmen und Kontrollzellen.